# SO101 Pick-and-Place — Behavior Cloning Training
**Runtime:** GPU (T4 recommended) | **Colab:** Runtime → Change runtime type → T4 GPU

In [ ]:
!pip install torch numpy mujoco --quiet
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

## 1. Upload 
Run the cell below and upload  from your local  folder.

In [ ]:
from google.colab import files
uploaded = files.upload()  # select demos.pkl
!ls -lh demos.pkl

## 2. Define Policy & Training

In [ ]:
import os, pickle, numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

class PolicyMLP(nn.Module):
    def __init__(self, in_dim=13, h1=128, h2=128, h3=64, out_dim=6):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, h1), nn.ReLU(),
            nn.Linear(h1, h2),    nn.ReLU(),
            nn.Linear(h2, h3),    nn.ReLU(),
            nn.Linear(h3, out_dim),
        )
    def forward(self, x): return self.net(x)

def load_demos(path):
    with open(path, "rb") as f:
        data = pickle.load(f)
    obs = np.array([t[0] for t in data], dtype=np.float32)
    act = np.array([t[1] for t in data], dtype=np.float32)
    return obs, act

obs, act = load_demos("demos.pkl")
print(f"Demos: obs={obs.shape}, act={act.shape}")

In [ ]:
dataset = TensorDataset(torch.from_numpy(obs), torch.from_numpy(act))
loader  = DataLoader(dataset, batch_size=32, shuffle=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

policy = PolicyMLP().to(device)
optimizer = optim.Adam(policy.parameters(), lr=1e-3)
criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)

best_loss = float("inf")
best_state = None

for epoch in range(1, 101):
    policy.train()
    epoch_loss = 0.0
    for batch_obs, batch_act in loader:
        batch_obs = batch_obs.to(device)
        batch_act = batch_act.to(device)
        pred = policy(batch_obs)
        loss = criterion(pred, batch_act)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * batch_obs.size(0)
    epoch_loss /= len(dataset)
    scheduler.step()
    if epoch_loss < best_loss:
        best_loss  = epoch_loss
        best_state = policy.state_dict().copy()
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | Loss: {epoch_loss:.6f} | Best: {best_loss:.6f}")

policy.load_state_dict(best_state)
torch.save(policy.state_dict(), "policy.pt")
print(f"
Training done! Best loss: {best_loss:.6f}")
print("Download policy.pt: ")
files.download("policy.pt")

## Architecture
